# 07 – Baseline en een eerste model op alle data

| | |
|---|---|
| **Auteur(s)** | Kobe Vandenberk |
| **Wat** | Bepalen wat "goed" is op alle data (2013–2026): hoe ver geraak je met gewone gemiddelden? Daarna een eerste, ongetuned gradient-boostingmodel en een eerste versie van de deployment, terwijl we in `08` verder tunen. |
| **Input** | `data/model_table.parquet` (uit `04_prepare_data`) |
| **Output** | `models/metrics/baseline*.json`, `models/metrics/quick_hgb.json`, `models/predictions/*.parquet`, `deploy/citi_bike_demand/model.pkl` |
| **GenAI** | Code en tekst opgesteld met hulp van Claude; keuzes en resultaten nagekeken door Kobe. |

## Hoe meten we op alle data?
- **Target:** aantal vertrekken per zone per uur (een telling ≥ 0).
- **Testset:** de **laatste 12 maanden** van de data (zie `04_prepare_data`). Die gebruiken we per model maar **één keer**, op het einde.
- **Validatie tijdens het kiezen/tunen: backtesting** (AWS-hoofdstuk 4). Drie folds: trainen op alle jaren vóór een validatieperiode van 12 maanden en valideren op die periode; de laatste fold valideert op de 12 maanden net vóór de testset. Een model ziet zo nooit de toekomst, net zoals in het echt.

  *Waarom niet meer de `GroupKFold` op weken uit de 2024-studie?* Met 13 jaar data is er een sterke **trend** (het netwerk groeit van 330 naar 2.200 stations). Met willekeurige weken zou het model voor 2019 leren van 2018 én 2020; in de praktijk voorspelt het altijd vooruit.
- **Metrics:** MAE (primair), RMSE, R², Poisson-deviance, zoals in de 2024-studie.

## 0. Imports en data

In [1]:
import json
import shutil
import subprocess
import sys

import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

import citibike as cb

table = cb.load_model_table()
X_train, y_train, _, X_test, y_test = cb.train_test(table)
train_time = table.loc[table["split"] == "train", "time"]
folds = cb.backtest_folds(train_time)
print(f"train: {len(X_train):,} zone-uren ({train_time.min():%Y-%m} t/m {train_time.max():%Y-%m}) | "
      f"test: {len(X_test):,} zone-uren ({table.loc[table['split'] == 'test', 'time'].min():%Y-%m} t/m "
      f"{table['time'].max():%Y-%m})")
for i, (tr, va) in enumerate(folds):
    print(f"  fold {i}: train {train_time.iloc[tr].min():%Y-%m}–{train_time.iloc[tr].max():%Y-%m} "
          f"({len(tr):,}) -> valideer {train_time.iloc[va].min():%Y-%m}–{train_time.iloc[va].max():%Y-%m} ({len(va):,})")
X_train.head()

train: 2,556,912 zone-uren (2013-06 t/m 2025-09) | test: 262,080 zone-uren (2025-10 t/m 2026-09)
  fold 0: train 2013-06–2022-09 (1,775,088) -> valideer 2022-10–2023-09 (255,504)
  fold 1: train 2013-06–2023-09 (2,030,592) -> valideer 2023-10–2024-09 (263,520)
  fold 2: train 2013-06–2024-09 (2,294,112) -> valideer 2024-10–2025-09 (262,800)


,zone,hour,weekday,month,is_weekend,is_holiday,temperature_c,precipitation_mm,wind_kmh,year,active_stations,snow_depth_cm
0,0,0,5,6,1,0,22.7,0.0,11.2,2013,32,0.0
1,0,1,5,6,1,0,21.9,0.0,10.5,2013,32,0.0
2,0,2,5,6,1,0,21.2,0.0,10.5,2013,32,0.0
3,0,3,5,6,1,0,20.6,0.0,9.7,2013,32,0.0
4,0,4,5,6,1,0,20.1,0.0,9.7,2013,32,0.0


## 1. Baselines met gemiddelden
| Baseline | Voorspelling |
|---|---|
| globaal | het gemiddelde over alle zones en uren |
| per zone × uur × weekend, **alle jaren** | het gemiddelde dagritme van elke zone over 13 jaar |
| per zone × uur × weekend, **laatste 12 maanden** | "hoeveel vertrekken er vorig jaar gemiddeld in deze zone om 8u op een weekdag?" |

De laatste is de echte lat: hij kent het dagpatroon van elke zone en het huidige niveau van het netwerk, maar niet het weer, het seizoen of feestdagen.

In [2]:
baselines = {
    "globaal gemiddelde": cb.GroupMeanRegressor(keys=()),
    "zone x uur x weekend, alle jaren": cb.GroupMeanRegressor(keys=("zone", "hour", "is_weekend")),
    "zone x uur x weekend, laatste 12 maanden": cb.GroupMeanRegressor(keys=("zone", "hour", "is_weekend"), recent_months=12),
}
rows = []
for name, model in baselines.items():
    cv = cb.cv_scores(model, X_train, y_train, cv=folds)
    rows.append({"baseline": name, "cv_mae": cv["cv_mae_mean"], **{f"fold {i}": m for i, m in enumerate(cv["cv_mae"])}})
pd.DataFrame(rows).set_index("baseline")

,cv_mae,fold 0,fold 1,fold 2
baseline,,,,
globaal gemiddelde,119.9848,102.2524,121.6872,136.0148
"zone x uur x weekend, alle jaren",82.0695,64.2797,85.7869,96.1419
"zone x uur x weekend, laatste 12 maanden",57.6031,47.9278,61.2460,63.6354


**Interpretatie:**
- Het **globale gemiddelde** zit 120 vertrekken per uur naast.
- **Zone × uur × weekend over alle jaren** helpt (82), maar minder dan in de 2024-studie: het gemiddelde over 13 jaar ligt veel lager dan de vraag van vandaag (het netwerk is 7 keer groter). Een gemiddelde over het verleden onderschat de toekomst systematisch.
- **Hetzelfde gemiddelde over alleen de laatste 12 maanden** is duidelijk beter (58): recente data zegt meer over volgend jaar dan oude data. Dat is een eerste aanwijzing dat "meer data" niet vanzelf beter is; het model moet de groei kunnen volgen (`year`, `active_stations`).
- De fout groeit van fold 0 (validatie 2022–2023: 48) naar fold 2 (2024–2025: 64): er zijn elk jaar meer vertrekken, dus ook grotere absolute fouten.

## 2. De beste baseline op de testset

In [3]:
reference = baselines["zone x uur x weekend, laatste 12 maanden"]
cv = cb.cv_scores(reference, X_train, y_train, cv=folds)
pred = reference.fit(X_train, y_train).predict(X_test)
test_metrics = cb.evaluate(y_test, pred)
metrics = {"notebook": "07_baseline_all_data", "model": "Baseline: zone x uur x weekend, laatste 12 maanden",
           **cv, **{f"test_{k}": v for k, v in test_metrics.items()}}
cb.save_results("baseline", metrics, table.loc[y_test.index, ["zone", "time", "trips"]].assign(pred=pred))
test_metrics

{'mae': 70.964, 'rmse': 115.4286, 'r2': 0.6802, 'poisson_deviance': 47.6182}

## 3. Snel eerste model: gradient boosting met standaardinstellingen
De techniek die in de 2024-studie won (`05e`, `06`): `HistGradientBoostingRegressor` met Poisson-verlies en de zone als categorie, nu met twee extra kenmerken (`year` en `active_stations`, zie `02b` §7). Eerst zonder tuning.

In [4]:
quick = HistGradientBoostingRegressor(loss="poisson", categorical_features=[cb.FEATURES.index("zone")],
                                      random_state=cb.SEED)
cv = cb.cv_scores(quick, X_train, y_train, cv=folds)
pred = quick.fit(X_train, y_train).predict(X_test)
test_metrics = cb.evaluate(y_test, pred)
metrics = {"notebook": "07_baseline_all_data", "model": "HistGradientBoosting (Poisson, standaard), alle data",
           "params": {}, **cv, **{f"test_{k}": v for k, v in test_metrics.items()}}
cb.save_results("quick_hgb", metrics, table.loc[y_test.index, ["zone", "time", "trips"]].assign(pred=pred))
pd.DataFrame({"baseline": json.loads((cb.METRICS_DIR / "baseline.json").read_text()),
              "snel model": metrics}).loc[["cv_mae", "cv_mae_mean", "test_mae", "test_rmse", "test_r2", "test_poisson_deviance"]]

,baseline,snel model
cv_mae,"[47.9278, 61.246, 63.6354]","[31.079, 41.2178, 36.8022]"
cv_mae_mean,57.6031,36.3663
test_mae,70.964,36.0537
test_rmse,115.4286,63.8321
test_r2,0.6802,0.9022
test_poisson_deviance,47.6182,14.9121


**Interpretatie:** het ongetunede boosting-model met de nieuwe kenmerken (`year`, `active_stations`, `snow_depth_cm`) halveert bijna de fout van de beste baseline: backtest-MAE 36,4 tegenover 57,6, en op de testset 36,1 tegenover 71,0 (R² 0,90 tegenover 0,68).

Opvallend: de baseline doet het op de testset veel slechter dan in de backtest (71 tegenover 58). Hij neemt aan dat volgend jaar hetzelfde is als vorig jaar, terwijl in de testperiode nieuwe stations bijkwamen (vooral in de Bronx en Queens, zie `02b` §7) en de winter koud en sneeuwrijk was. Het boosting-model ziet het aantal actieve stations en het weer en blijft daardoor even goed op de testset (36,1) als in de backtest (36,4).

## 4. Eerste versie van de deployment
`train.py` traint het model voor de API en schrijft het naar `deploy/citi_bike_demand/model.pkl`. Het leest de hyperparameters uit `models/best_params.json`. Als vertrekpunt nemen we de **getunede waarden uit de 2024-studie** (`models/2024/best_params.json`); `08` tunet ze daarna opnieuw op alle data en overschrijft dit bestand.

`--force`: het vorige model werd op een andere testset gemeten (2024), dus een vergelijking is niet zinvol.

In [5]:
shutil.copyfile(cb.MODEL_2024 / "best_params.json", cb.MODEL_DIR / "best_params.json")
result = subprocess.run([sys.executable, "train.py", "--force"], capture_output=True, text=True)
print(result.stdout[-1500:], result.stderr[-1500:])

Data: 2,818,992 zone-uren (2,556,912 train / 262,080 test: 2025-10-01 t/m 2026-09-30), 30 zones
Hold-out: {'mae': 29.0563, 'rmse': 53.8429, 'r2': 0.9304, 'poisson_deviance': 10.6337}
Bundle geschreven naar C:\Users\kobev\Documents\semester 1\3APPAI - Deep Learning\CloudAI-Challenge\deploy\citi_bike_demand\model.pkl (4.8 MB)
 


## Samenvatting
| Model | Backtest-MAE | Test-MAE | Test-R² |
|---|---|---|---|
| globaal gemiddelde | 120,0 | | |
| zone × uur × weekend, alle jaren | 82,1 | | |
| **baseline**: zone × uur × weekend, laatste 12 maanden | 57,6 | 71,0 | 0,68 |
| HistGradientBoosting, standaardinstellingen | 36,4 | 36,1 | 0,90 |
| HistGradientBoosting, hyperparameters uit 2024 (via `train.py`) | | **29,1** | **0,93** |

- Met de hyperparameters uit de 2024-studie haalt het model op alle data al **MAE 29,1 en R² 0,93**: veel beter dan de standaardinstellingen. Die stoppen na 100 bomen; op 2,5 miljoen rijen heeft het model er duidelijk meer nodig (2024: tot 1.056 bomen met een kleine learning rate).
- Dit model staat nu in `deploy/citi_bike_demand/model.pkl`. In `08` tunen we opnieuw op alle data.